In [ ]:
from utils import initialize_environment
hf_token = initialize_environment()

import pandas as pd
from data import DataProcessor
from preprocessing import DataAnalyzer  
from features import BoW, TFIDF, Word2VecExtractor, PreTrainedWord2Vec
from models import LogisticClassifier, NaiveBayesClassifier, SVMClassifier
from trainer_utils import ModelTrainer

# 1. Φόρτωση και προετοιμασία των baseline δεδομένων για τα μοντέλα
X, y, encoders, test_indices = DataProcessor().prepare_baseline_data()

# 2. Φόρτωση των ακατέργαστων δεδομένων αποκλειστικά για την παραγωγή διαγραμμάτων (EDA)
processor = DataProcessor()
raw_data = processor.process_data(
    train_path='csv/train.csv', 
    val_path='csv/valid.csv', 
    test_path='csv/test.csv', 
    tokenize=False
)
train_df = raw_data['train_df']

# Καθολική λίστα για τη δυναμική αποθήκευση ΟΛΩΝ των αποτελεσμάτων
all_results = []

Τρέχων φάκελος εργασίας: c:\Users\Alekos\Desktop\NLP
Προειδοποίηση: Δεν βρέθηκε HF_TOKEN στο αρχείο .env


In [ ]:
# ΠΑΡΑΓΩΓΗ ΔΙΑΓΡΑΜΜΑΤΩΝ ΚΑΤΑΝΟΜΗΣ 

# Αρχικοποίηση του Analyzer
analyzer = DataAnalyzer()

# 1. Παραγωγή και εμφάνιση του διαγράμματος για τις κατηγορίες κινδύνου
print("\n[ Οπτικοποίηση: Κατανομή των Hazard Categories ]")
analyzer.plot_hazard_distribution(train_df)

# 2. Παραγωγή και εμφάνιση του διαγράμματος για τις κατηγορίες προϊόντων (Long Tail)
print("\n[ Οπτικοποίηση: Κατανομή των Product Categories ]")
analyzer.plot_product_distribution(train_df)

In [ ]:
# LOGISTIC REGRESSION 

# 1. Μοντέλο με Bag of Words (BoW)
print("\n[ Εκπαίδευση: Logistic Regression + BoW ]")
trainer_log_bow = ModelTrainer(extractor=BoW(max_features=5000, ngram_range=(1, 2)), classifier_class=LogisticClassifier)
st1_log_bow = trainer_log_bow.run_experiment(X, y, encoders, test_indices, export_name="log_reg_bow")
all_results.append({"Model": "Logistic Regression", "Feature": "BoW", "ST1 Score": st1_log_bow})

# 2. Μοντέλο με TF-IDF
print("\n[ Εκπαίδευση: Logistic Regression + TF-IDF ]")
trainer_log_tfidf = ModelTrainer(extractor=TFIDF(max_features=5000, ngram_range=(1, 2)), classifier_class=LogisticClassifier)
st1_log_tfidf = trainer_log_tfidf.run_experiment(X, y, encoders, test_indices, export_name="log_reg_tfidf")
all_results.append({"Model": "Logistic Regression", "Feature": "TF-IDF", "ST1 Score": st1_log_tfidf})

# 3. Μοντέλο με Custom Word2Vec (From Scratch)
print("\n[ Εκπαίδευση: Logistic Regression + Custom Word2Vec ]")
trainer_log_w2v_custom = ModelTrainer(extractor=Word2VecExtractor(vector_size=100), classifier_class=LogisticClassifier)
st1_log_w2v_custom = trainer_log_w2v_custom.run_experiment(X, y, encoders, test_indices, export_name="log_reg_w2v_custom")
all_results.append({"Model": "Logistic Regression", "Feature": "Custom Word2Vec", "ST1 Score": st1_log_w2v_custom})

# 4. Μοντέλο με Pre-trained Word2Vec (Google News)
print("\n[ Εκπαίδευση: Logistic Regression + Google Word2Vec ]")
trainer_log_w2v_pre = ModelTrainer(extractor=PreTrainedWord2Vec(), classifier_class=LogisticClassifier)
st1_log_w2v_pre = trainer_log_w2v_pre.run_experiment(X, y, encoders, test_indices, export_name="log_reg_w2v_pretrained")
all_results.append({"Model": "Logistic Regression", "Feature": "Google Word2Vec", "ST1 Score": st1_log_w2v_pre})


[ Εκπαίδευση: Logistic Regression + BoW ]


KeyboardInterrupt: 

In [ ]:
# NAIVE BAYES 

# 1. Μοντέλο με Bag of Words (BoW)
print("\n[ Εκπαίδευση: Naive Bayes + BoW ]")
trainer_nb_bow = ModelTrainer(extractor=BoW(max_features=5000, ngram_range=(1, 2)), classifier_class=NaiveBayesClassifier)
st1_nb_bow = trainer_nb_bow.run_experiment(X, y, encoders, test_indices, export_name="naive_bayes_bow")
all_results.append({"Model": "Naive Bayes", "Feature": "BoW", "ST1 Score": st1_nb_bow})

# 2. Μοντέλο με TF-IDF (Default Alpha = 1.0)
print("\n[ Εκπαίδευση: Naive Bayes + TF-IDF (Default) ]")
trainer_nb_tfidf = ModelTrainer(extractor=TFIDF(max_features=5000, ngram_range=(1, 2)), classifier_class=NaiveBayesClassifier)
st1_nb_tfidf = trainer_nb_tfidf.run_experiment(X, y, encoders, test_indices, export_name="naive_bayes_tfidf")
all_results.append({"Model": "Naive Bayes", "Feature": "TF-IDF", "ST1 Score": st1_nb_tfidf})

# 3. Hyperparameter Tuning: Δοκιμές τιμών Alpha με αναπαράσταση TF-IDF
alpha_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
print("\n[ Έναρξη Tuning για την παράμετρο Alpha ]")
for a in alpha_values:
    print(f"-> Εκπαίδευση Naive Bayes με alpha = {a}")
    trainer = ModelTrainer(
        extractor=TFIDF(max_features=5000, ngram_range=(1, 2)), 
        classifier_class=lambda: NaiveBayesClassifier(alpha=a)
    )
    score = trainer.run_experiment(X, y, encoders, test_indices, export_name=f"naive_bayes_tfidf_alpha_{a}")
    all_results.append({"Model": f"Naive Bayes (alpha={a})", "Feature": "TF-IDF", "ST1 Score": score})

=== LOGISTIC REGRESSION ===
 Macro-F1 Score: 0.7076

              precision    recall  f1-score   support

           0       0.96      0.96      0.96       207
           1       0.98      0.95      0.97       194
           2       0.77      0.82      0.79        28
           3       0.33      0.50      0.40         2
           4       0.91      0.98      0.95        63
           5       0.70      0.73      0.71        41
           7       0.67      0.50      0.57         8
           8       0.67      0.57      0.62        14
           9       0.43      0.38      0.40         8

    accuracy                           0.91       565
   macro avg       0.71      0.71      0.71       565
weighted avg       0.91      0.91      0.91       565

 Macro-F1 Score: 0.6726

              precision    recall  f1-score   support

           0       0.60      0.86      0.71         7
           1       0.69      0.69      0.69        75
           2       0.73      0.73      0.73        15


In [ ]:
#SUPPORT VECTOR MACHINE (SVM) 

# 1. Μοντέλο με Bag of Words (BoW)
print("\n[ Εκπαίδευση: SVM + BoW ]")
trainer_svm_bow = ModelTrainer(extractor=BoW(max_features=5000, ngram_range=(1, 2)), classifier_class=SVMClassifier)
st1_svm_bow = trainer_svm_bow.run_experiment(X, y, encoders, test_indices, export_name="svm_bow")
all_results.append({"Model": "SVM (Default)", "Feature": "BoW", "ST1 Score": st1_svm_bow})

# 2. Μοντέλο με TF-IDF (Default C = 1.0)
print("\n[ Εκπαίδευση: SVM + TF-IDF (Default) ]")
trainer_svm_tfidf = ModelTrainer(extractor=TFIDF(max_features=5000, ngram_range=(1, 2)), classifier_class=SVMClassifier)
st1_svm_tfidf = trainer_svm_tfidf.run_experiment(X, y, encoders, test_indices, export_name="svm_tfidf")
all_results.append({"Model": "SVM (Default)", "Feature": "TF-IDF", "ST1 Score": st1_svm_tfidf})

# 3. Hyperparameter Tuning: Δοκιμές τιμών κανονικοποίησης C με αναπαράσταση TF-IDF
c_values = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
print("\n[ Έναρξη Tuning για την παράμετρο C ]")
for c in c_values:
    print(f"-> Εκπαίδευση SVM με C = {c}")
    trainer = ModelTrainer(
        extractor=TFIDF(max_features=5000, ngram_range=(1, 2)), 
        classifier_class=lambda: SVMClassifier(C=c)
    )
    score = trainer.run_experiment(X, y, encoders, test_indices, export_name=f"svm_tfidf_c_{c}")
    all_results.append({"Model": f"SVM (C={c})", "Feature": "TF-IDF", "ST1 Score": score})

=== NAIVE BAYES ===

--- Naive Bayes Evaluation ---
Macro-F1 Score: 0.4268


--- Naive Bayes Evaluation ---
Macro-F1 Score: 0.3106


 ΤΕΛΙΚΟ ST1 SCORE (naive_bayes_bow): 0.3687 <---
Το αρχείο 'csv/submission_naive_bayes_bow.csv' δημιουργήθηκε επιτυχώς (997 γραμμές).

--- Naive Bayes Evaluation ---
Macro-F1 Score: 0.4696


--- Naive Bayes Evaluation ---
Macro-F1 Score: 0.3100


 ΤΕΛΙΚΟ ST1 SCORE (naive_bayes_tfidf): 0.3898 <---
Το αρχείο 'csv/submission_naive_bayes_tfidf.csv' δημιουργήθηκε επιτυχώς (997 γραμμές).


In [ ]:

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(by="ST1 Score", ascending=False).reset_index(drop=True)

print(results_df.to_string(index=False))

best_model = results_df.iloc[0]
print(f"\n Καλυτερο Κλασικό Μοντέλο: {best_model['Model']} με αναπαράσταση {best_model['Feature']} (Score: {best_model['ST1 Score']:.4f})")

=== ΜΑΓΕΙΡΕΜΑ: NAIVE BAYES + TF-IDF (Alpha Tuning) ===

[ Εκπαίδευση Naive Bayes με alpha = 0.001 ]

--- Naive Bayes Evaluation ---
Macro-F1 Score: 0.5271


--- Naive Bayes Evaluation ---
Macro-F1 Score: 0.4088


 ΤΕΛΙΚΟ ST1 SCORE (naive_bayes_tfidf_alpha_0.001): 0.4679 <---
Το αρχείο 'csv/submission_naive_bayes_tfidf_alpha_0.001.csv' δημιουργήθηκε επιτυχώς (997 γραμμές).

[ Εκπαίδευση Naive Bayes με alpha = 0.01 ]

--- Naive Bayes Evaluation ---
Macro-F1 Score: 0.4624


--- Naive Bayes Evaluation ---
Macro-F1 Score: 0.3649


 ΤΕΛΙΚΟ ST1 SCORE (naive_bayes_tfidf_alpha_0.01): 0.4137 <---
Το αρχείο 'csv/submission_naive_bayes_tfidf_alpha_0.01.csv' δημιουργήθηκε επιτυχώς (997 γραμμές).

[ Εκπαίδευση Naive Bayes με alpha = 0.05 ]

--- Naive Bayes Evaluation ---
Macro-F1 Score: 0.4630


--- Naive Bayes Evaluation ---
Macro-F1 Score: 0.3433


 ΤΕΛΙΚΟ ST1 SCORE (naive_bayes_tfidf_alpha_0.05): 0.4032 <---
Το αρχείο 'csv/submission_naive_bayes_tfidf_alpha_0.05.csv' δημιουργήθηκε επιτυχώς (997 γ